# fase_5 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 5.

**Purpose**: Migrasi data Rapor dan Penilaian dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import re
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

Connected to dataleap_v5_example and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('format_rapor', 'rapor_format'),
    ('format_rapor_detil', 'rapor_format_sub'),
    ('format_rapor_rumus', 'rapor_format_formula'),
    ('format_rapor_detil_rumus', 'rapor_format_formula_sub'),
    ('format_raport_level', 'rapor_level_config'),
    ('rapor', 'rapor_siswa'),
    ('file_rapor_siswa', 'rapor_siswa_file'),
    ('history_rapor', 'rapor_lacak')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    try:
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()
        print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")
    except Exception as e:
        print(f"❌ ERROR loading {old_t}: {e}")

✅ format_rapor loaded: 45 records
✅ format_rapor_detil loaded: 129 records
✅ format_rapor_rumus loaded: 3 records
✅ format_rapor_detil_rumus loaded: 1650 records
✅ format_raport_level loaded: 348 records
✅ rapor loaded: 22837 records
✅ file_rapor_siswa loaded: 1506 records
✅ history_rapor loaded: 1366 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# --- TRANSFORMATION ---

# 1. format_rapor -> rapor_format (+ urutan dari import CSV)
if 'format_rapor' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor'])
    mapping = {
        'idformat_rapor': 'id_rapor_format',
        'idpendkursus': 'id_kursus', 'title': 'judul_rapor'
    }
    df_rf = df.rename(columns=mapping)[list(mapping.values())]
    # Merge kolom urutan dari rapor_format_import.csv (sudah diurutkan manual)
    df_urutan_rf = pd.read_csv('rapor_format_import.csv')[['judul_rapor', 'urutan']]
    df_rf = df_rf.merge(df_urutan_rf, on='judul_rapor', how='left')
    df_rf['urutan'] = df_rf['urutan'].astype('Int64')  # cegah float karena NaN
    transformed_dfs['rapor_format'] = df_rf

# 2. format_rapor_detil -> rapor_format_sub (+ urutan dari import CSV)
if 'format_rapor_detil' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_detil'])
    mapping = {
        'idformat_rd': 'id_rapor_format_sub',
        'idformat_rapor': 'id_rapor_format', 'subtitle': 'sub_judul_rapor'
    }
    df_rfs = df.rename(columns=mapping)[list(mapping.values())]
    # Merge kolom urutan dari rapor_format_sub_import.csv (sudah diurutkan manual)
    # id_rapor_format di KEDUA sisi pakai format string yg sama (misal 'F00001') -> join langsung
    df_urutan_rfs = pd.read_csv('rapor_format_sub_import.csv')[['id_rapor_format', 'sub_judul_rapor', 'urutan']]
    df_rfs = df_rfs.merge(df_urutan_rfs, on=['id_rapor_format', 'sub_judul_rapor'], how='left')
    df_rfs['urutan'] = df_rfs['urutan'].astype('Int64')  # cegah float karena NaN
    transformed_dfs['rapor_format_sub'] = df_rfs

# 3. format_rapor_rumus -> rapor_format_formula
if 'format_rapor_rumus' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_rumus'])
    mapping = {
        'idfrr': 'id_rapor_format_formula',
        'idformat_rapor': 'id_rapor_format', 'param_operator': 'logika_operator'
    }
    transformed_dfs['rapor_format_formula'] = df.rename(columns=mapping)[list(mapping.values())]

# 4. format_rapor_detil_rumus -> rapor_format_formula_sub
if 'format_rapor_detil_rumus' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_detil_rumus'])
    mapping = {
        'idfrdr': 'id_rapor_format_formula_sub',
        'idformat_rd': 'id_rapor_format_sub', 'param_operator': 'logika_operator',
        'idlevel': 'id_level'
    }
    transformed_dfs['rapor_format_formula_sub'] = df.rename(columns=mapping)[list(mapping.values())]

# 5. format_raport_level -> rapor_level_config
if 'format_raport_level' in raw_data:
    df = pd.DataFrame(raw_data['format_raport_level'])
    mapping = {
        'idformat_rl': 'id_rapor_level_config', 'idlevel': 'id_level',
        'idpendkursus': 'id_kursus', 'idformat_rapor': 'id_rapor_format'
    }
    transformed_dfs['rapor_level_config'] = df.rename(columns=mapping)[list(mapping.values())]

# 6. rapor_sub_level (Tabel Baru)
transformed_dfs['rapor_sub_level'] = pd.DataFrame(columns=['id_rapor_sub_level', 'id_rapor_format_sub', 'id_level'])

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# Helper extract_int in Fase 5
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

# 7. rapor -> rapor_siswa
if 'rapor' in raw_data:
    df = pd.DataFrame(raw_data['rapor'])
    
    # Generate integer ID auto-increment mapping
    df = df.reset_index()
    df['id_rapor_siswa_new'] = df['index'] + 1
    rapor_id_map = dict(zip(df['idrapor'], df['id_rapor_siswa_new']))
    df['id_rapor_siswa'] = df['id_rapor_siswa_new']
    
    df['id_siswa_clean'] = df['idsiswa'].apply(extract_int).astype('Int64')
    df['id_jadwal_clean'] = df['idjadwal'].apply(extract_int).astype('Int64')
    
    # Map idp_nilai string (e.g. 'P00745') to new parameter_nilai auto-increment ID
    cursor_old.execute("SELECT idp_nilai FROM parameter_nilai ORDER BY idp_nilai")
    param_rows = cursor_old.fetchall()
    param_map = {}
    for idx, row in enumerate(param_rows):
        if isinstance(row, dict):
            param_map[row['idp_nilai']] = idx + 1
        elif isinstance(row, (list, tuple)):
            param_map[row[0]] = idx + 1
    df['id_parameter_nilai'] = df['idp_nilai'].map(param_map).astype('Int64')
    
    mapping = {
        'id_rapor_siswa': 'id_rapor_siswa', 'id_jadwal_clean': 'id_jadwal', 'id_siswa_clean': 'id_siswa',
        'tanggal': 'tanggal_input', 'id_parameter_nilai': 'id_parameter_nilai', 'nilai': 'final_result'
    }
    transformed_dfs['rapor_siswa'] = df.rename(columns=mapping)[list(mapping.values())]

# 8. file_rapor_siswa -> rapor_siswa_file
if 'file_rapor_siswa' in raw_data and 'rapor_siswa' in transformed_dfs:
    df = pd.DataFrame(raw_data['file_rapor_siswa'])
    
    # Generate integer ID auto-increment mapping for file table
    df = df.reset_index()
    df['id_rapor_siswa_file_new'] = df['index'] + 1
    file_id_map = dict(zip(df['idfile'], df['id_rapor_siswa_file_new']))
    df['id_rapor_siswa_file'] = df['id_rapor_siswa_file_new']
    
    # Fetch old idrapor string and map it to new id_rapor_siswa integer
    df_rapor_old = pd.DataFrame(raw_data['rapor'])[['idsiswa', 'idjadwal', 'idrapor']].drop_duplicates(subset=['idsiswa', 'idjadwal'])
    df = df.merge(df_rapor_old, on=['idsiswa', 'idjadwal'], how='left')
    df['id_rapor_siswa'] = df['idrapor'].map(rapor_id_map).astype('Int64')
    
    mapping = {
        'id_rapor_siswa_file': 'id_rapor_siswa_file', 'id_rapor_siswa': 'id_rapor_siswa', 'path': 'file_rapor_path'
    }
    transformed_dfs['rapor_siswa_file'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 9. history_rapor -> rapor_lacak
if 'history_rapor' in raw_data and 'rapor_siswa_file' in transformed_dfs:
    df = pd.DataFrame(raw_data['history_rapor'])
    df['status'] = df['status'].replace({'Terkirim': 'Terkirim', 'Gagal': 'Gagal'})
    
    df_file_old = pd.DataFrame(raw_data['file_rapor_siswa'])[['idfile', 'idsiswa', 'idjadwal']]
    df_file_old['id_rapor_siswa_file'] = df_file_old['idfile'].map(file_id_map).astype('Int64')
    
    df['id_siswa_clean'] = df['idsiswa'].apply(extract_int).astype('Int64')
    df['id_jadwal_clean'] = df['idjadwal'].apply(extract_int).astype('Int64')
    
    df_merged = df.merge(df_file_old[['idsiswa', 'idjadwal', 'id_rapor_siswa_file']], on=['idsiswa', 'idjadwal'], how='left')
    df_merged['id_rapor_siswa_file'] = df_merged['id_rapor_siswa_file'].astype('Int64')
    df_merged['id_rapor_lacak'] = df_merged['idhistori'].apply(extract_int).astype('Int64')
    
    mapping = {
        'id_rapor_lacak': 'id_rapor_lacak', 'id_siswa_clean': 'id_siswa',
        'id_jadwal_clean': 'id_jadwal', 'tgl': 'tanggal_terkirim', 'status': 'status_pengiriman'
    }
    transformed_dfs['rapor_lacak'] = df_merged.rename(columns=mapping)[list(mapping.values()) + ['id_rapor_siswa_file']]

print(f"OK: Transformasi {len(transformed_dfs)} tabel Fase 5 selesai.")


OK: Transformasi 9 tabel Fase 5 selesai.


## 3.1 Verifikasi Hasil Transformasi
Bagian ini menampilkan perbandingan jumlah data dan tipe data untuk pengecekan manual.

In [5]:
# 3.1.1 Ringkasan Jumlah Baris
print("📊 RINGKASAN MIGRASI (RECORDS COUNT)")
print("="*70)
summary_list = []
for old_t, new_t in hanif_tables_map:
    old_c = len(raw_data.get(old_t, []))
    new_c = len(transformed_dfs.get(new_t, []))
    summary_list.append({
        'Tabel Lama': old_t,
        'Tabel Baru': new_t,
        'Old Recs': old_c,
        'New Recs': new_c,
        'Diff': new_c - old_c,
        'Status': "✅ OK" if old_c == new_c else "⚠️ Cek"
    })
display(pd.DataFrame(summary_list))

total_old_all = sum(len(records) for records in raw_data.values())
total_new_all = sum(len(df) for df in transformed_dfs.values())
print(f"\n📢 TOTAL REKAPITULASI: {total_old_all} (Old) ➔ {total_new_all} (New)")
if total_old_all == total_new_all: print("✅ SEMUA DATA TERANGKUT")
else: print(f"⚠️ ADA SELISIH: {total_new_all - total_old_all} baris")

📊 RINGKASAN MIGRASI (RECORDS COUNT)


,Tabel Lama,Tabel Baru,Old Recs,New Recs,Diff,Status
0,format_rapor,rapor_format,45,285,240,⚠️ Cek
1,format_rapor_detil,rapor_format_sub,129,129,0,✅ OK
2,format_rapor_rumus,rapor_format_formula,3,3,0,✅ OK
3,format_rapor_detil_rumus,rapor_format_formula_sub,1650,1650,0,✅ OK
4,format_raport_level,rapor_level_config,348,348,0,✅ OK
5,rapor,rapor_siswa,22837,22837,0,✅ OK
6,file_rapor_siswa,rapor_siswa_file,1506,1506,0,✅ OK
7,history_rapor,rapor_lacak,1366,1366,0,✅ OK



📢 TOTAL REKAPITULASI: 27884 (Old) ➔ 28124 (New)
⚠️ ADA SELISIH: 240 baris


In [6]:
# 3.1.2 Output Pengecekan Kolom Spesifik (KETERANGAN mapping.md)
print("\n🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)")
print("="*70)

# 1. Pengecekan Tanggal Input (Rapor Siswa)
if 'rapor_siswa' in transformed_dfs:
    print("\n[RAPOR_SISWA] Pengecekan tanggal_input (Direct Mapping):")
    display(transformed_dfs['rapor_siswa'][['id_rapor_siswa', 'tanggal_input']].head(5))

# 2. Pengecekan Rapor Lacak (Enum Status)
if 'rapor_lacak' in transformed_dfs:
    print("\n[RAPOR_LACAK] Pengecekan Normalisasi Status Pengiriman:")
    display(transformed_dfs['rapor_lacak']['status_pengiriman'].value_counts())

# 3. Pengecekan Rapor Siswa File
if 'rapor_siswa_file' in transformed_dfs:
    print("\n[RAPOR_SISWA_FILE] Pengecekan path file:")
    display(transformed_dfs['rapor_siswa_file'][['id_rapor_siswa', 'file_rapor_path']].head(5))


🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)

[RAPOR_SISWA] Pengecekan tanggal_input (Direct Mapping):


,id_rapor_siswa,tanggal_input
0,1,2023-09-29 15:01:39
1,2,2023-09-29 15:01:39
2,3,2023-09-29 15:01:39
3,4,2023-09-29 15:01:39
4,5,2023-09-29 15:01:39



[RAPOR_LACAK] Pengecekan Normalisasi Status Pengiriman:


status_pengiriman
Terkirim    1366
Name: count, dtype: int64


[RAPOR_SISWA_FILE] Pengecekan path file:


,id_rapor_siswa,file_rapor_path
0,5166,uploads/rapor/S0000329.jpeg
1,5172,uploads/rapor/S0000474.jpeg
2,5183,uploads/rapor/S0000481.jpeg
3,5178,uploads/rapor/S0000475.jpeg
4,5189,uploads/rapor/S0000482.jpeg


In [7]:
# 3.1.3 Detail Perbandingan Kolom & Tipe Data (Side-by-Side)
print("\n🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE")
for old_t, new_t in hanif_tables_map:
    print(f"\n{'='*15} {old_t.upper()} ➔ {new_t.upper()} {'='*15}")
    
    df_old = pd.DataFrame(raw_data.get(old_t, []))
    df_new = transformed_dfs.get(new_t, pd.DataFrame())
    
    if not df_new.empty or not df_old.empty:
        comparison = []
        table_mapping = {}
        if old_t == 'format_rapor': table_mapping = {'idformat_rapor': 'id_rapor_format', 'idpendkursus': 'id_kursus', 'title': 'judul_rapor'}
        elif old_t == 'format_rapor_detil': table_mapping = {'idformat_rd': 'id_rapor_format_sub', 'idformat_rapor': 'id_rapor_format', 'subtitle': 'sub_judul_rapor'}
        elif old_t == 'format_rapor_rumus': table_mapping = {'idfrr': 'id_rapor_format_formula', 'idformat_rapor': 'id_rapor_format', 'param_operator': 'logika_operator'}
        elif old_t == 'format_rapor_detil_rumus': table_mapping = {'idfrdr': 'id_rapor_format_formula_sub', 'idformat_rd': 'id_rapor_format_sub', 'param_operator': 'logika_operator', 'idlevel': 'id_level'}
        elif old_t == 'format_raport_level': table_mapping = {'idformat_rl': 'id_rapor_level_config', 'idlevel': 'id_level', 'idpendkursus': 'id_kursus', 'idformat_rapor': 'id_rapor_format'}
        elif old_t == 'rapor': table_mapping = {'idrapor': 'id_rapor_siswa', 'idjadwal': 'id_jadwal', 'idsiswa': 'id_siswa', 'tanggal': 'tanggal_input', 'idp_nilai': 'id_parameter_nilai', 'nilai': 'final_result'}
        elif old_t == 'file_rapor_siswa': table_mapping = {'idfile': 'id_rapor_siswa_file', 'idsiswa': 'id_rapor_siswa', 'path': 'file_rapor_path'}
        elif old_t == 'history_rapor': table_mapping = {'idhistori': 'id_rapor_lacak', 'idsiswa': 'id_siswa', 'idjadwal': 'id_jadwal', 'tgl': 'tanggal_terkirim', 'status': 'status_pengiriman'}

        for old_col, new_col in table_mapping.items():
            comparison.append({
                'Old Column': old_col,
                'Old Type': str(df_old[old_col].dtype) if not df_old.empty and old_col in df_old.columns else "N/A",
                '➔': '➔',
                'New Column': new_col,
                'New Type': str(df_new[new_col].dtype) if not df_new.empty and new_col in df_new.columns else "N/A"
            })
        
        # Cek kolom baru
        if not df_new.empty:
            for col in df_new.columns:
                if col not in table_mapping.values():
                    comparison.append({
                        'Old Column': '(KOLOM BARU / CUSTOM)',
                        'Old Type': '-',
                        '➔': '➔',
                        'New Column': col,
                        'New Type': str(df_new[col].dtype)
                    })
        
        display(pd.DataFrame(comparison))
        if not df_new.empty:
            print(f"\n--- SAMPLE DATA NEW (2 Baris) ---")
            display(df_new.head(2))
    else:
        print(f"⚠️ Tabel {new_t} kosong.")


🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE

=============== FORMAT_RAPOR ➔ RAPOR_FORMAT ===============


,Old Column,Old Type,➔,New Column,New Type
0,idformat_rapor,object,➔,id_rapor_format,object
1,idpendkursus,object,➔,id_kursus,object
2,title,object,➔,judul_rapor,object
3,(KOLOM BARU / CUSTOM),-,➔,urutan,Int64



--- SAMPLE DATA NEW (2 Baris) ---


,id_rapor_format,id_kursus,judul_rapor,urutan
0,F00001,K00001,CLASSROOM ASSESSMENT,1
1,F00001,K00001,CLASSROOM ASSESSMENT,1



=============== FORMAT_RAPOR_DETIL ➔ RAPOR_FORMAT_SUB ===============


...

,id_rapor_siswa_file,id_rapor_siswa,file_rapor_path
0,1,5166,uploads/rapor/S0000329.jpeg
1,2,5172,uploads/rapor/S0000474.jpeg



=============== HISTORY_RAPOR ➔ RAPOR_LACAK ===============


,Old Column,Old Type,➔,New Column,New Type
0,idhistori,object,➔,id_rapor_lacak,Int64
1,idsiswa,object,➔,id_siswa,Int64
2,idjadwal,object,➔,id_jadwal,Int64
3,tgl,datetime64[ns],➔,tanggal_terkirim,datetime64[ns]
4,status,object,➔,status_pengiriman,object
5,(KOLOM BARU / CUSTOM),-,➔,id_rapor_siswa_file,Int64



--- SAMPLE DATA NEW (2 Baris) ---


,id_rapor_lacak,id_siswa,id_jadwal,tanggal_terkirim,status_pengiriman,id_rapor_siswa_file
0,1,609,173,2024-11-18 10:57:59,Terkirim,396
1,5,144,352,2024-12-02 11:37:11,Terkirim,399


## 4. Export ke Pickle

In [8]:
file_name = 'fase_5_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

total_records_new = sum(len(df) for df in transformed_dfs.values())
total_records_old = sum(len(records) for records in raw_data.values())

migration_result = {
    'fase': 'fase_5',
    'script': 'script_hanif',
    'fase_num': 5,
    'status': 'ready_for_insert',
    'old_records_total': total_records_old,
    'new_records_total': total_records_new,
    'diff': total_records_new - total_records_old,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_5",
  "script": "script_hanif",
  "fase_num": 5,
  "status": "ready_for_insert",
  "old_records_total": 27884,
  "new_records_total": 28124,
  "diff": 240,
  "pickle_file": "fase_5_hanif.pkl",
  "timestamp": "2026-06-10T00:16:23.034557"
}


In [9]:
# --- EXPORT KE CSV UNTUK VERIFIKASI ---
EXPORT_TO_CSV = True  # Ubah ke False jika tidak ingin menghasilkan file CSV

if EXPORT_TO_CSV:
    import os
    import pandas as pd
    target_dir = "../extract/cek_csv"
    os.makedirs(target_dir, exist_ok=True)
    for tbl_name, df_tbl in transformed_dfs.items():
        csv_path = os.path.join(target_dir, f"{tbl_name}.csv")
        df_to_save = df_tbl.copy()
        
        # Clean any float ID/FK columns that contain .0 to pure integers
        for col in df_to_save.columns:
            col_lower = col.lower()
            is_id_col = col_lower.startswith('id_') or col_lower.endswith('_id') or col_lower == 'id' or 'id_' in col_lower or '_id_' in col_lower
            if is_id_col:
                non_nulls = df_to_save[col].dropna()
                if not non_nulls.empty:
                    try:
                        pd.to_numeric(non_nulls, errors='raise')
                        df_to_save[col] = pd.to_numeric(df_to_save[col], errors='coerce').round().astype('Int64')
                    except (ValueError, TypeError):
                        pass
        
        # Fix: Convert any StringDtype to object for clean serialization
        for col in df_to_save.columns:
            if str(df_to_save[col].dtype) in ['string', 'string[python]']:
                df_to_save[col] = df_to_save[col].astype(object)
        df_to_save.to_csv(csv_path, index=False)
        print(f"💾 Tabel {tbl_name} diekspor ke {csv_path} ({len(df_tbl)} baris)")
else:
    print("ℹ️ Ekspor ke CSV dinonaktifkan.")

💾 Tabel rapor_format diekspor ke ../extract/cek_csv\rapor_format.csv (285 baris)
💾 Tabel rapor_format_sub diekspor ke ../extract/cek_csv\rapor_format_sub.csv (129 baris)
💾 Tabel rapor_format_formula diekspor ke ../extract/cek_csv\rapor_format_formula.csv (3 baris)
💾 Tabel rapor_format_formula_sub diekspor ke ../extract/cek_csv\rapor_format_formula_sub.csv (1650 baris)
💾 Tabel rapor_level_config diekspor ke ../extract/cek_csv\rapor_level_config.csv (348 baris)
💾 Tabel rapor_sub_level diekspor ke ../extract/cek_csv\rapor_sub_level.csv (0 baris)
💾 Tabel rapor_siswa diekspor ke ../extract/cek_csv\rapor_siswa.csv (22837 baris)
💾 Tabel rapor_siswa_file diekspor ke ../extract/cek_csv\rapor_siswa_file.csv (1506 baris)
💾 Tabel rapor_lacak diekspor ke ../extract/cek_csv\rapor_lacak.csv (1366 baris)
